# 04 | Ensembles and feature–label circularity
**Purpose.** Evaluate the three manuscript ensemble strategies (stacking, hard voting, probability averaging) and the four feature configurations examined in the circularity ablation. All tests reuse the **same 7,000 original held-out Gaia sources**.

**Prerequisite:** notebook 03 with CatBoost, MLP, and LightGBM trained in each feature configuration.  
**Outputs:** `ensemble_metrics.csv`, `feature_ablation.csv`, saved predictions and figures.

**Interpretation:** agreement with the Gaia-derived `SpType-ELS` labels, not independent spectroscopic classification accuracy.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "config" / "experiment.json").exists():
    ROOT = ROOT.parent
if not (ROOT / "config" / "experiment.json").exists():
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks/ folder.")
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display
from src.gaia_pipeline import *
cfg = load_config(ROOT)
np.random.seed(cfg["random_state"])
print("Repository:", ROOT)
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize
balanced = read_object(ROOT, "preprocessed.joblib")["balanced"]
benchmark = read_object(ROOT, "benchmark.joblib")
train_idx, test_idx = benchmark["train_idx"], benchmark["test_idx"]
y_train, y_test = benchmark["y_train"], benchmark["y_test"]
classes = np.asarray(benchmark["classes"])
assert len(test_idx) == len(y_test)
for feature_name in FEATURE_SETS:
    assert all(name in benchmark["by_feature"][feature_name]["predictions"] for name in BASE_MODELS), f"Missing base learner for {feature_name}"
FIG = output_dir(ROOT) / "figures"; FIG.mkdir(exist_ok=True)

## Out-of-fold stacking, hard voting and probability averaging
The meta-classifier sees only out-of-fold base probabilities from five training folds. Hard voting uses three class predictions, with the mean of posterior probabilities used only to resolve ties. Vote fractions, not calibrated probabilities, are used for its diagnostic curves.

In [ ]:
combined = benchmark["by_feature"]["combined_features"]
X_train = balanced.iloc[train_idx][FEATURE_SETS["combined_features"]]
X_test = balanced.iloc[test_idx][FEATURE_SETS["combined_features"]]
n_classes = len(classes)
cv = StratifiedKFold(n_splits=cfg["cv_folds"], shuffle=True, random_state=cfg["random_state"])
oof_probabilities = {}
for name in BASE_MODELS:
    print("Stacking out-of-fold:", name)
    oof = np.zeros((len(train_idx), n_classes))
    for tr, va in cv.split(X_train, y_train):
        pipe = make_base_pipeline(name, cfg)
        pipe.fit(X_train.iloc[tr], y_train[tr])
        oof[va] = pipe.predict_proba(X_train.iloc[va])
    oof_probabilities[name] = oof
meta_X = np.hstack([oof_probabilities[name] for name in BASE_MODELS])
meta_test = np.hstack([combined["probabilities"][name] for name in BASE_MODELS])
meta_model = LogisticRegression(max_iter=5000, solver="lbfgs", random_state=cfg["random_state"])
meta_model.fit(meta_X, y_train)
stack_pred, stack_prob = meta_model.predict(meta_test), meta_model.predict_proba(meta_test)
hard_pred, hard_vote = hard_voting_from_arrays([combined["predictions"][n] for n in BASE_MODELS], [combined["probabilities"][n] for n in BASE_MODELS], n_classes)
avg_prob = np.mean([combined["probabilities"][n] for n in BASE_MODELS], axis=0)
avg_pred = avg_prob.argmax(axis=1)
ensemble_predictions = {ENSEMBLE_NAMES["stacking"]: stack_pred, ENSEMBLE_NAMES["hard_voting"]: hard_pred, ENSEMBLE_NAMES["probability_average"]: avg_pred}
ensemble_probabilities = {ENSEMBLE_NAMES["stacking"]: stack_prob, ENSEMBLE_NAMES["hard_voting"]: hard_vote, ENSEMBLE_NAMES["probability_average"]: avg_prob}
ensemble_table = pd.DataFrame([{"Model": name, **metric_dict(y_test, prediction)} for name, prediction in ensemble_predictions.items()])
save_table(ensemble_table, ROOT, "ensemble_metrics.csv")
save_object({"predictions": ensemble_predictions, "probabilities": ensemble_probabilities}, ROOT, "ensembles.joblib")
joblib.dump(meta_model, output_dir(ROOT) / "models" / "stacking_meta.joblib")
display(ensemble_table.round(4))

## Held-out class diagnostics
Reproduce the seven-class hard-voting confusion matrix, one-vs-rest ROC curves, and precision–recall curves. The class order is the encoded order produced by `LabelEncoder`, which is preserved throughout the benchmark.

In [ ]:
model_name = ENSEMBLE_NAMES["hard_voting"]
cm = confusion_matrix(y_test, ensemble_predictions[model_name], labels=np.arange(n_classes))
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="viridis", xticklabels=classes, yticklabels=classes, ax=ax)
ax.set(xlabel="Predicted label", ylabel="Gaia catalogue label", title="Hard voting: held-out confusion matrix")
fig.tight_layout(); fig.savefig(FIG / "hard_voting_confusion.png"); plt.show()
y_bin = label_binarize(y_test, classes=np.arange(n_classes))
for kind in ("roc", "pr"):
    fig, ax = plt.subplots(figsize=(7, 5))
    for i, cls in enumerate(classes):
        score = ensemble_probabilities[model_name][:, i]
        if kind == "roc":
            x, y, _ = roc_curve(y_bin[:, i], score)
            label = f"{cls} (AUC={auc(x, y):.3f})"
        else:
            precision, recall, _ = precision_recall_curve(y_bin[:, i], score)
            x, y = recall, precision
            label = f"{cls} (AP={average_precision_score(y_bin[:, i], score):.3f})"
        ax.plot(x, y, label=label)
    ax.set(xlabel="False positive rate" if kind == "roc" else "Recall", ylabel="True positive rate" if kind == "roc" else "Precision", title=f"Hard voting: {kind.upper()} curves")
    ax.legend(); fig.tight_layout(); fig.savefig(FIG / f"hard_voting_{kind}.png"); plt.show()
print(classification_report(y_test, hard_pred, target_names=classes, digits=4))

## Observed-photometry ablation
The observed baseline contains **only** G, BP and RP apparent magnitudes and their direct colour differences. The feature ablation retains the original FLAME-complete cohort to keep the test-source membership identical, even though the baseline's predictors contain no FLAME variables. Consequently, the observed-photometry result does not remove the cohort-selection effect.

In [ ]:
X_obs_train = balanced.iloc[train_idx][OBSERVED_FEATURES]
X_obs_test = balanced.iloc[test_idx][OBSERVED_FEATURES]
observed_predictions, observed_probabilities = {}, {}
for name in BASE_MODELS:
    print("Observed photometry:", name)
    pipe = make_base_pipeline(name, cfg)
    pipe.fit(X_obs_train, y_train)
    observed_predictions[name] = np.asarray(pipe.predict(X_obs_test)).ravel().astype(int)
    observed_probabilities[name] = pipe.predict_proba(X_obs_test)
    joblib.dump(pipe, output_dir(ROOT) / "models" / f"observed_{name}.joblib")
observed_hard_pred, _ = hard_voting_from_arrays([observed_predictions[n] for n in BASE_MODELS], [observed_probabilities[n] for n in BASE_MODELS], n_classes)
assert np.array_equal(hard_pred, hard_voting_from_arrays([combined["predictions"][n] for n in BASE_MODELS], [combined["probabilities"][n] for n in BASE_MODELS], n_classes)[0])
ablation_cases = [
    ("CatBoost", "Observed photometry", observed_predictions["CatBoost"]),
    ("CatBoost", "Corrected photometry", benchmark["by_feature"]["photometric_features"]["predictions"]["CatBoost"]),
    ("CatBoost", "Inferred parameters", benchmark["by_feature"]["inferred_astrophysical_parameters"]["predictions"]["CatBoost"]),
    ("CatBoost", "Combined features", combined["predictions"]["CatBoost"]),
    ("Hard Voting", "Observed photometry", observed_hard_pred),
    ("Hard Voting", "Corrected photometry", hard_voting_from_arrays([benchmark["by_feature"]["photometric_features"]["predictions"][n] for n in BASE_MODELS], [benchmark["by_feature"]["photometric_features"]["probabilities"][n] for n in BASE_MODELS], n_classes)[0]),
    ("Hard Voting", "Inferred parameters", hard_voting_from_arrays([benchmark["by_feature"]["inferred_astrophysical_parameters"]["predictions"][n] for n in BASE_MODELS], [benchmark["by_feature"]["inferred_astrophysical_parameters"]["probabilities"][n] for n in BASE_MODELS], n_classes)[0]),
    ("Hard Voting", "Combined features", hard_pred)
]
ablation_table = pd.DataFrame([{"Model": name, "Features": setting, **metric_dict(y_test, pred)} for name, setting, pred in ablation_cases])
ablation_table["delta_f1_vs_observed"] = ablation_table["f1_macro"] - ablation_table.groupby("Model")["f1_macro"].transform("first")
save_table(ablation_table, ROOT, "feature_ablation.csv")
display(ablation_table.round(4))

## Feature-ablation visualization
Differences are reported as changes in macro-F1 on the same held-out catalogue labels. These comparisons are **not** causal estimates of circularity in Gaia inference pipelines.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=ablation_table, x="Features", y="f1_macro", hue="Model", order=["Observed photometry", "Corrected photometry", "Inferred parameters", "Combined features"], errorbar=None, ax=ax)
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=2)
ax.set(xlabel="Feature configuration", ylabel="Macro-F1", ylim=(0, 1), title="Feature ablation: agreement with Gaia catalogue labels")
fig.tight_layout(); fig.savefig(FIG / "feature_ablation.png"); plt.show()